# M5 Forecasting Accuracy — Preprocess (Colab)

CSV → df_features.parquet → train/val/eval 分割

| ステップ | 内容 |
|----------|------|
| 1. Setup | 環境検出・データ準備 |
| 2. Phase 1 | CSV → df_features.parquet (ストリーミング) |
| 3. Phase 1.5 | Target Encoding (store_id × dept_id lag-28) |
| 4. Phase 2 | parquet → train_X.dat, train_cat.dat, val.parquet, eval.parquet |

## Step 0: Setup

In [ ]:
# ============================================================
# SETUP — Colab / Local 両対応
# ============================================================
import sys, os, subprocess
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")

# pip install (Colab のみ)
if IS_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])

COMPETITION = 'm5-forecasting-accuracy'
DATA_FILES  = ['calendar.csv', 'sales_train_evaluation.csv', 'sell_prices.csv']

def has_all_files(d: Path) -> bool:
    return d.exists() and all((d / f).exists() for f in DATA_FILES)

# ============================================================
# Drive マウント (Colab のみ — 出力先に使用)
# ============================================================
_drive_ok = False
if IS_COLAB:
    _drive_ok = Path('/content/drive/MyDrive').exists()
    if not _drive_ok:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            _drive_ok = True
        except Exception as _e:
            print(f'Drive mount failed: {_e}')

# ============================================================
# DATA_DIR 検出 (CSV 読み込み元)
# ============================================================
DATA_DIR = None

if IS_COLAB:
    for c in [Path(f'/content/{COMPETITION}'), Path('/content/data')]:
        if has_all_files(c):
            DATA_DIR = c
            break
    if DATA_DIR is None and _drive_ok:
        for c in [
            Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}'),
            Path(f'/content/drive/MyDrive/{COMPETITION}'),
        ]:
            if has_all_files(c):
                DATA_DIR = c
                break
    if DATA_DIR is None:
        # CSV がどこにもない → ダウンロード
        DATA_DIR = Path(f'/content/{COMPETITION}')
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        _token = None
        _token_paths = [
            Path('/content/drive/MyDrive/.kaggle/access_token'),
            Path('/content/drive/MyDrive/kaggle/access_token'),
        ]
        for _tp in _token_paths:
            if _tp.exists():
                _token = _tp.read_text().strip()
                break
        if _token is None:
            try:
                from google.colab import userdata
                _token = userdata.get('KAGGLE_TOKEN')
            except Exception:
                pass
        if _token is None:
            raise RuntimeError(
                'Kaggle token が見つかりません。\n'
                'Drive に .kaggle/access_token を配置するか、\n'
                'Colab Secrets に KAGGLE_TOKEN を設定してください'
            )
        import requests, zipfile
        _url = f'https://www.kaggle.com/api/v1/competitions/data/download-all/{COMPETITION}'
        print('Downloading...')
        with requests.get(_url, headers={'Authorization': f'Bearer {_token}'},
                          stream=True, allow_redirects=True) as _r:
            _r.raise_for_status()
            _zip = DATA_DIR / f'{COMPETITION}.zip'
            with open(_zip, 'wb') as _f:
                for _chunk in _r.iter_content(1024*1024):
                    _f.write(_chunk)
        with zipfile.ZipFile(_zip) as _z:
            _z.extractall(DATA_DIR)
        _zip.unlink()
        del _token
        print('Download complete')
else:
    for c in [
        Path(f'/mnt/c/Users/hiroyuki_nakayama/src/kaggle-with-mcp/{COMPETITION}'),
        Path('.'),
    ]:
        if has_all_files(c):
            DATA_DIR = c
            break

assert DATA_DIR is not None and has_all_files(DATA_DIR), \
    f'CSV ファイルが見つかりません: {DATA_DIR}'

# ============================================================
# OUTPUT_DIR (前処理結果の書き出し先)
#   Colab → Google Drive (ランタイム切断後も永続化)
#   Local → DATA_DIR と同じ
# ============================================================
if IS_COLAB and _drive_ok:
    OUTPUT_DIR = Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    OUTPUT_DIR = DATA_DIR

print(f'DATA_DIR   : {DATA_DIR}  (CSV 読み込み元)')
print(f'OUTPUT_DIR : {OUTPUT_DIR}  (前処理結果の書き出し先)')
for f in DATA_FILES:
    print(f'  {f:<40} {(DATA_DIR/f).stat().st_size/1e6:6.1f} MB')

## Step 1 (Phase 1): CSV → df_features.parquet

In [ ]:
# ============================================================
# Phase 1: CSV → df_features.parquet (ストリーミング)
# ============================================================
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# --- 定数 ---
PRED_HORIZON = 28
N_DAYS       = 1941
KEEP_FROM_DAY = 1100  # メモリ不足なら 1300 に上げる
CHUNK_SIZE    = 1000
PRICES_CHUNK  = 500_000

META_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
CAL_COLS  = ['d', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
             'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
             'snap_CA', 'snap_TX', 'snap_WI']
CAT_COLS  = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id',
             'weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

OUT_PATH    = OUTPUT_DIR / 'df_features.parquet'
PRICES_PATH = DATA_DIR / 'sell_prices.csv'
SALES_PATH  = DATA_DIR / 'sales_train_evaluation.csv'
CAL_PATH    = DATA_DIR / 'calendar.csv'

# --- ユーティリティ (preprocess.py と同一) ---
def reduce_mem(df):
    for col in df.select_dtypes('integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes('float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

def stream_item_max_price(prices_path):
    result = {}
    for chunk in pd.read_csv(prices_path, chunksize=PRICES_CHUNK,
                             usecols=['item_id', 'sell_price']):
        for item_id, price in zip(chunk['item_id'], chunk['sell_price']):
            if item_id not in result or price > result[item_id]:
                result[item_id] = price
    return result

def stream_prices_for_items(prices_path, item_ids, store_ids):
    parts = []
    for chunk in pd.read_csv(prices_path, chunksize=PRICES_CHUNK):
        mask = chunk['item_id'].isin(item_ids) & chunk['store_id'].isin(store_ids)
        filtered = chunk[mask]
        if len(filtered) > 0:
            parts.append(filtered)
    if parts:
        return reduce_mem(pd.concat(parts, ignore_index=True))
    return pd.DataFrame(columns=['store_id', 'item_id', 'wm_yr_wk', 'sell_price'])

def process_chunk(chunk, d_cols, calendar, prices_path, item_max_price, keep_from_day):
    mat = chunk[d_cols].values
    has_sale = (mat > 0)
    first_idx = has_sale.argmax(axis=1)
    never_sold_mask = has_sale.sum(axis=1) == 0
    fsd = pd.DataFrame({
        'id': chunk['id'].values,
        'first_sale_day': np.where(never_sold_mask, N_DAYS + 1, first_idx + 1),
    })
    del mat, has_sale, first_idx, never_sold_mask

    df = chunk.melt(id_vars=META_COLS, value_vars=d_cols, var_name='d', value_name='sales')
    df['d_num'] = df['d'].str[2:].astype('int16')
    df['sales'] = df['sales'].astype('float32')
    df = df[df['d_num'] >= keep_from_day].reset_index(drop=True)
    df = df.merge(fsd, on='id', how='left')
    df = df[df['d_num'] >= df['first_sale_day']].drop('first_sale_day', axis=1).reset_index(drop=True)
    del fsd
    if len(df) == 0:
        return df

    df = df.merge(calendar, on='d', how='left')
    prices = stream_prices_for_items(prices_path, set(df['item_id'].unique()), set(df['store_id'].unique()))
    df = df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
    del prices
    df = df.sort_values(['id', 'd_num']).reset_index(drop=True)

    df['price_change_rel'] = df.groupby('id')['sell_price'].pct_change().astype('float32')
    mp = df['item_id'].map(item_max_price).values.astype('float32')
    df['discount_ratio'] = ((mp - df['sell_price'].values) / np.where(mp == 0, np.nan, mp)).astype('float32')
    del mp

    # === SNAP 交差特徴量 (カテゴリエンコード前: state_id が文字列) ===
    _snap_map = {'CA': 'snap_CA', 'TX': 'snap_TX', 'WI': 'snap_WI'}
    df['snap_active'] = np.int8(0)
    for _state, _col in _snap_map.items():
        _m = df['state_id'] == _state
        df.loc[_m, 'snap_active'] = df.loc[_m, _col].values.astype('int8')
    df['snap_wday'] = (df['snap_active'] * df['wday']).astype('int8')

    for col in CAT_COLS:
        df[col] = df[col].astype('category').cat.codes.astype('int16')
    df['is_weekend'] = (df['wday'] >= 6).astype('int8')
    df['day'] = df['date'].dt.day.astype('int8')
    df['is_month_end'] = df['date'].dt.is_month_end.astype('int8')
    df['is_month_start'] = df['date'].dt.is_month_start.astype('int8')
    df['is_christmas_nearby'] = ((df['date'].dt.month == 12) & (df['date'].dt.day.between(23, 27))).astype('int8')

    for lag in [28, 35, 42, 56]:
        df[f'lag_{lag}'] = df.groupby('id')['sales'].shift(lag).astype('float32')
    df['_s28'] = df.groupby('id')['sales'].shift(PRED_HORIZON).astype('float32')
    for win in [7, 28, 56]:
        grp = df.groupby('id')['_s28']
        df[f'roll_mean_{win}'] = grp.rolling(win, min_periods=1).mean().reset_index(level=0, drop=True).astype('float32')
        df[f'roll_std_{win}'] = grp.rolling(win, min_periods=1).std().reset_index(level=0, drop=True).astype('float32')
    # roll_median_7: 外れ値に頑健な中央値
    df['roll_median_7'] = (
        df.groupby('id')['_s28']
          .rolling(7, min_periods=1).median()
          .reset_index(level=0, drop=True).astype('float32')
    )
    # EWMA: 指数平滑移動平均 (直近トレンド捕捉)
    df['ewma_7'] = (
        df.groupby('id')['_s28']
          .ewm(span=7, adjust=False).mean()
          .reset_index(level=0, drop=True).astype('float32')
    )
    df['ewma_28'] = (
        df.groupby('id')['_s28']
          .ewm(span=28, adjust=False).mean()
          .reset_index(level=0, drop=True).astype('float32')
    )

    _zero = (df['_s28'] == 0).astype('float32')
    df['zeros_last_28'] = _zero.groupby(df['id']).rolling(28, min_periods=1).sum().reset_index(level=0, drop=True).astype('float32')
    _mask = df['sales'] > 0
    df['_last_sale_day'] = df['d_num'].where(_mask).astype('float32')
    df['_last_sale_day'] = df.groupby('id')['_last_sale_day'].ffill()
    df['days_since_last_sale'] = (
        df.groupby('id')['d_num'].shift(PRED_HORIZON)
        - df.groupby('id')['_last_sale_day'].shift(PRED_HORIZON)
    ).astype('float32').fillna(0)
    df.drop(columns=['_s28', '_last_sale_day'], inplace=True)
    return reduce_mem(df)

# --- 実行 ---
if OUT_PATH.exists():
    pf = pq.ParquetFile(OUT_PATH)
    print(f'[SKIP] {OUT_PATH.name} が既に存在 ({pf.metadata.num_rows:,} rows)')
    del pf
else:
    t0 = time.time()
    calendar = reduce_mem(pd.read_csv(CAL_PATH, parse_dates=['date'])[CAL_COLS])
    # イベント前後3日フラグ
    calendar = calendar.sort_values('d').reset_index(drop=True)
    _has_ev = calendar['event_name_1'].notna().astype(float)
    calendar['event_nearby'] = (
        _has_ev.rolling(7, center=True, min_periods=1).max().astype('int8')
    )
    del _has_ev

    item_max_price = stream_item_max_price(PRICES_PATH)
    header = pd.read_csv(SALES_PATH, nrows=0)
    d_cols = [c for c in header.columns if c.startswith('d_')]
    print(f'calendar: {calendar.shape}, items: {len(item_max_price)}, days: {len(d_cols)}')

    reader = pd.read_csv(SALES_PATH, chunksize=CHUNK_SIZE)
    writer = None
    total_rows = 0
    for i, chunk in enumerate(reader):
        df_chunk = process_chunk(chunk, d_cols, calendar, PRICES_PATH, item_max_price, KEEP_FROM_DAY)
        if len(df_chunk) == 0:
            continue
        table = pa.Table.from_pandas(df_chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(OUT_PATH), table.schema)
        writer.write_table(table)
        total_rows += len(df_chunk)
        print(f'  chunk {i+1:>3} | rows {total_rows:>12,} | {time.time()-t0:.0f}s')
        del df_chunk, table
    if writer:
        writer.close()
    print(f'\nPhase 1 完了: {total_rows:,} rows ({OUT_PATH.stat().st_size/1e6:.0f} MB, {(time.time()-t0)/60:.1f} min)')

## Step 1.5 (Phase 1.5): Target Encoding + snap_uplift_store

In [ ]:
# ============================================================
# Phase 1.5: Store Profiling + Target Encoding (train期間のみ)
# ============================================================
VAL_START_DAY_15 = 1886
PAYDAY_DAYS = {14, 15, 16, 28, 29, 30, 31}
HOBBIES_CAT_ID = 1   # cat_id エンコード (sorted): FOODS=0, HOBBIES=1, HOUSEHOLD=2
HOUSEHOLD_CAT_ID = 2

pf = pq.ParquetFile(OUT_PATH)
existing_cols = [f.name for f in pf.schema_arrow]
new_cols = ['snap_dependency_score', 'payroll_dependency_score',
            'weekend_intensity', 'luxury_affinity_score',
            'snap_dep_interaction', 'weekend_interaction',
            'price_sensitivity_index', 'pb_ratio',
            'price_x_psi', 'snap_x_pb',
            'cat_income_elasticity', 'stockpiling_index',
            'roll_mean_56_weighted',
            'income_event_sensitivity', 'spike_hint']

if all(c in existing_cols for c in new_cols):
    print(f'[SKIP] 店舗プロファイリング特徴量が既に存在')
    del pf
else:
    t0 = time.time()
    n_rg = pf.metadata.num_row_groups

    # === Pass 0: PB_Ratio 用に店舗ごとの価格 P20 閾値を算出 ===
    print('[0] 店舗別 P20 価格閾値を算出中...')
    store_prices = {}
    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=['store_id', 'd_num', 'sell_price']).to_pandas()
        rg_tr = rg[rg['d_num'].values < VAL_START_DAY_15]
        for sid, price in zip(rg_tr['store_id'].values, rg_tr['sell_price'].values):
            p = float(price)
            if p > 0 and not np.isnan(p):
                sid_int = int(sid)
                if sid_int not in store_prices:
                    store_prices[sid_int] = []
                store_prices[sid_int].append(p)
        del rg, rg_tr
    p20_threshold = {}
    for sid, prices_list in store_prices.items():
        p20_threshold[sid] = float(np.percentile(prices_list, 20))
    del store_prices
    print(f'  P20 thresholds: {dict(sorted(p20_threshold.items()))}')

    # === Pass 1: train期間のみで集計 ===
    print(f'[1/3] 集計中 (d_num < {VAL_START_DAY_15} のみ)...')
    te_agg = {}
    snap_agg = {}
    payroll_agg = {}
    weekend_agg = {}
    luxury_agg = {}
    psi_agg = {}
    pb_agg = {}
    cie_agg = {}
    stockpile_daily = {}
    resid_agg = {}  # (store_id, cat_id, segment) → [residual_sum, count]

    has_snap_active = 'snap_active' in existing_cols
    has_is_weekend = 'is_weekend' in existing_cols
    base_cols = ['store_id', 'dept_id', 'd_num', 'sales', 'day', 'cat_id',
                 'discount_ratio', 'sell_price', 'roll_mean_28']
    if has_snap_active:
        base_cols.append('snap_active')
    else:
        base_cols.extend(['state_id', 'snap_CA', 'snap_TX', 'snap_WI'])
        print('    (snap_active 列なし → snap_CA/TX/WI から再計算)')
    if has_is_weekend:
        base_cols.append('is_weekend')
    else:
        base_cols.append('wday')
        print('    (is_weekend 列なし → wday から再計算)')
    read_cols = list(dict.fromkeys(base_cols))

    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=read_cols).to_pandas()
        if not has_snap_active:
            rg['snap_active'] = np.int8(0)
            rg.loc[rg['state_id'] == 0, 'snap_active'] = rg.loc[rg['state_id'] == 0, 'snap_CA'].values.astype('int8')
            rg.loc[rg['state_id'] == 1, 'snap_active'] = rg.loc[rg['state_id'] == 1, 'snap_TX'].values.astype('int8')
            rg.loc[rg['state_id'] == 2, 'snap_active'] = rg.loc[rg['state_id'] == 2, 'snap_WI'].values.astype('int8')
        if not has_is_weekend:
            rg['is_weekend'] = (rg['wday'] >= 6).astype('int8')
        mask = rg['d_num'].values < VAL_START_DAY_15
        rg_tr = rg[mask]
        # TE 集計
        for sid, did, dnum, sales in zip(
            rg_tr['store_id'].values, rg_tr['dept_id'].values,
            rg_tr['d_num'].values, rg_tr['sales'].values,
        ):
            key = (int(sid), int(did), int(dnum))
            if key in te_agg:
                te_agg[key][0] += float(sales)
                te_agg[key][1] += 1
            else:
                te_agg[key] = [float(sales), 1]
        # 指標の集計
        for sid, snap, sales, day, is_we, cat, dr, sp, dnum, rm28 in zip(
            rg_tr['store_id'].values, rg_tr['snap_active'].values,
            rg_tr['sales'].values, rg_tr['day'].values,
            rg_tr['is_weekend'].values, rg_tr['cat_id'].values,
            rg_tr['discount_ratio'].values, rg_tr['sell_price'].values,
            rg_tr['d_num'].values, rg_tr['roll_mean_28'].values,
        ):
            sid_int = int(sid)
            s = float(sales)
            # SNAP
            if sid_int not in snap_agg:
                snap_agg[sid_int] = [0.0, 0, 0.0, 0]
            if int(snap) == 1:
                snap_agg[sid_int][0] += s
                snap_agg[sid_int][1] += 1
            else:
                snap_agg[sid_int][2] += s
                snap_agg[sid_int][3] += 1
            # Payroll
            if sid_int not in payroll_agg:
                payroll_agg[sid_int] = [0.0, 0, 0.0, 0]
            payroll_agg[sid_int][2] += s
            payroll_agg[sid_int][3] += 1
            if int(day) in PAYDAY_DAYS:
                payroll_agg[sid_int][0] += s
                payroll_agg[sid_int][1] += 1
            # Weekend
            if sid_int not in weekend_agg:
                weekend_agg[sid_int] = [0.0, 0, 0.0, 0]
            if int(is_we) == 1:
                weekend_agg[sid_int][0] += s
                weekend_agg[sid_int][1] += 1
            else:
                weekend_agg[sid_int][2] += s
                weekend_agg[sid_int][3] += 1
            # Luxury (HOBBIES)
            if sid_int not in luxury_agg:
                luxury_agg[sid_int] = [0.0, 0.0]
            luxury_agg[sid_int][1] += s
            if int(cat) == HOBBIES_CAT_ID:
                luxury_agg[sid_int][0] += s
            # Price Sensitivity
            dr_f = float(dr)
            if not np.isnan(dr_f):
                if sid_int not in psi_agg:
                    psi_agg[sid_int] = [0.0, 0, 0.0, 0]
                if dr_f > 0.1:
                    psi_agg[sid_int][0] += s
                    psi_agg[sid_int][1] += 1
                else:
                    psi_agg[sid_int][2] += s
                    psi_agg[sid_int][3] += 1
            # PB Ratio
            sp_f = float(sp)
            if not np.isnan(sp_f) and sp_f > 0:
                if sid_int not in pb_agg:
                    pb_agg[sid_int] = [0.0, 0.0]
                pb_agg[sid_int][1] += s
                if sp_f <= p20_threshold.get(sid_int, 0.0):
                    pb_agg[sid_int][0] += s
            # category_income_elasticity
            cat_int = int(cat)
            cie_key = (sid_int, cat_int)
            if cie_key not in cie_agg:
                cie_agg[cie_key] = [0.0, 0, 0.0, 0]
            is_event_day = (int(snap) == 1) or (int(day) in PAYDAY_DAYS)
            if is_event_day:
                cie_agg[cie_key][0] += s
                cie_agg[cie_key][1] += 1
            else:
                cie_agg[cie_key][2] += s
                cie_agg[cie_key][3] += 1
            # stockpiling_index (HOUSEHOLD daily sales)
            if cat_int == HOUSEHOLD_CAT_ID:
                dnum_int = int(dnum)
                if sid_int not in stockpile_daily:
                    stockpile_daily[sid_int] = {}
                if dnum_int not in stockpile_daily[sid_int]:
                    stockpile_daily[sid_int][dnum_int] = 0.0
                stockpile_daily[sid_int][dnum_int] += s
            # 乖離率 (residual from roll_mean_28)
            rm28_f = float(rm28)
            if rm28_f > 0 and not np.isnan(rm28_f):
                resid = (s - rm28_f) / rm28_f
                if int(snap) == 1:
                    seg = 0
                elif int(day) in PAYDAY_DAYS:
                    seg = 1
                elif int(is_we) == 1:
                    seg = 2
                else:
                    seg = 3
                rkey = (sid_int, cat_int, seg)
                if rkey not in resid_agg:
                    resid_agg[rkey] = [0.0, 0]
                resid_agg[rkey][0] += resid
                resid_agg[rkey][1] += 1
        del rg, rg_tr

    te_lookup = {k: v[0] / v[1] for k, v in te_agg.items()}
    del te_agg
    print(f'  TE lookup entries: {len(te_lookup):,}')

    # --- 指標の算出 ---
    snap_dep = {}
    for sid, (s_sum, s_cnt, ns_sum, ns_cnt) in snap_agg.items():
        snap_avg = s_sum / s_cnt if s_cnt > 0 else 0.0
        nosnap_avg = ns_sum / ns_cnt if ns_cnt > 0 else 0.0
        snap_dep[sid] = (snap_avg / nosnap_avg) if nosnap_avg > 0 else 1.0
    del snap_agg

    payroll_dep = {}
    for sid, (pd_sum, pd_cnt, t_sum, t_cnt) in payroll_agg.items():
        pd_avg = pd_sum / pd_cnt if pd_cnt > 0 else 0.0
        t_avg = t_sum / t_cnt if t_cnt > 0 else 0.0
        payroll_dep[sid] = (pd_avg / t_avg) if t_avg > 0 else 1.0
    del payroll_agg

    we_intensity = {}
    for sid, (we_sum, we_cnt, wd_sum, wd_cnt) in weekend_agg.items():
        we_avg = we_sum / we_cnt if we_cnt > 0 else 0.0
        wd_avg = wd_sum / wd_cnt if wd_cnt > 0 else 0.0
        we_intensity[sid] = (we_avg / wd_avg) if wd_avg > 0 else 1.0
    del weekend_agg

    lux_affinity = {}
    for sid, (hob_sum, tot_sum) in luxury_agg.items():
        lux_affinity[sid] = (hob_sum / tot_sum) if tot_sum > 0 else 0.0
    del luxury_agg

    price_sens = {}
    for sid, (d_sum, d_cnt, n_sum, n_cnt) in psi_agg.items():
        d_avg = d_sum / d_cnt if d_cnt > 0 else 0.0
        n_avg = n_sum / n_cnt if n_cnt > 0 else 0.0
        price_sens[sid] = (d_avg / n_avg) if n_avg > 0 else 1.0
    del psi_agg

    pb_ratio = {}
    for sid, (pb_sum, tot_sum) in pb_agg.items():
        pb_ratio[sid] = (pb_sum / tot_sum) if tot_sum > 0 else 0.0
    del pb_agg

    cie_lookup = {}
    for key, (e_sum, e_cnt, n_sum, n_cnt) in cie_agg.items():
        e_avg = e_sum / e_cnt if e_cnt > 0 else 0.0
        n_avg = n_sum / n_cnt if n_cnt > 0 else 0.0
        cie_lookup[key] = (e_avg / n_avg) if n_avg > 0 else 1.0
    del cie_agg
    print(f'  CIE entries: {len(cie_lookup)} (store×cat)')

    stockpile_idx = {}
    for sid, daily in stockpile_daily.items():
        if len(daily) < 56:
            stockpile_idx[sid] = 0.0
            continue
        d_sorted = sorted(daily.keys())
        vals = np.array([daily[d] for d in d_sorted], dtype='float64')
        if vals.std() < 1e-8:
            stockpile_idx[sid] = 0.0
            continue
        n = len(vals)
        if n <= 28:
            stockpile_idx[sid] = 0.0
            continue
        mean = vals.mean()
        var = vals.var()
        autocov = np.mean((vals[:n-28] - mean) * (vals[28:] - mean))
        stockpile_idx[sid] = float(autocov / var) if var > 0 else 0.0
    del stockpile_daily
    print(f'  Stockpiling index: {dict(sorted(stockpile_idx.items()))}')

    # 乖離率の平均
    resid_mean = {}
    for rkey, (rsum, rcnt) in resid_agg.items():
        resid_mean[rkey] = rsum / rcnt if rcnt > 0 else 0.0
    del resid_agg

    # income_event_sensitivity
    SEG_NAMES = {0: 'SNAP', 1: 'Payday', 2: 'Weekend'}
    income_event_sens = {}
    spike_expected = {}
    all_store_ids_r = set()
    for (sid, cid, seg), val in resid_mean.items():
        all_store_ids_r.add(sid)
        spike_expected[(sid, cid, seg)] = val
    for sid in all_store_ids_r:
        seg_totals = [0.0, 0.0, 0.0]
        seg_counts = [0, 0, 0]
        for (s, c, seg), val in resid_mean.items():
            if s == sid and seg < 3:
                seg_totals[seg] += val
                seg_counts[seg] += 1
        seg_avgs = [seg_totals[i] / seg_counts[i] if seg_counts[i] > 0 else 0.0
                    for i in range(3)]
        best_seg = int(np.argmax(seg_avgs))
        income_event_sens[sid] = best_seg
    print(f'\n    Income Event Sensitivity:')
    for sid in sorted(income_event_sens.keys()):
        seg = income_event_sens[sid]
        snap_r = resid_mean.get((sid, 0, 0), 0.0)
        pay_r = resid_mean.get((sid, 0, 1), 0.0)
        we_r = resid_mean.get((sid, 0, 2), 0.0)
        print(f'      store {sid}: {SEG_NAMES[seg]:>8} '
              f'(SNAP:{snap_r:+.3f} Pay:{pay_r:+.3f} WE:{we_r:+.3f})')

    # --- クラスタリング ---
    store_ids = sorted(snap_dep.keys())
    snap_vals = np.array([snap_dep[s] for s in store_ids])
    pay_vals = np.array([payroll_dep[s] for s in store_ids])
    snap_z = (snap_vals - snap_vals.mean()) / (snap_vals.std() + 1e-8)
    pay_z = (pay_vals - pay_vals.mean()) / (pay_vals.std() + 1e-8)

    INCOME_LABELS = {0: 'Type-S', 1: 'Type-P', 2: 'Type-B'}
    store_income = {}
    print(f'\n    Store Profiling Results:')
    print(f'    {"store":>6} {"SNAP_dep":>9} {"Payroll":>8} {"Weekend":>8} '
          f'{"Luxury":>7} {"PSI":>6} {"PB%":>6} {"CIE":>5} {"Stock":>6} {"type":>8}')
    for idx, sid in enumerate(store_ids):
        sz, pz = snap_z[idx], pay_z[idx]
        if sz > 0 and sz >= pz:
            t = 0
        elif pz > 0 and pz > sz:
            t = 1
        else:
            t = 2
        store_income[sid] = t
        cie_foods = cie_lookup.get((sid, 0), 1.0)
        print(f'    {sid:>6} {snap_dep[sid]:>9.4f} {payroll_dep[sid]:>8.4f} '
              f'{we_intensity[sid]:>8.4f} {lux_affinity[sid]:>7.4f} '
              f'{price_sens[sid]:>6.3f} {pb_ratio[sid]:>6.3f} '
              f'{cie_foods:>5.3f} {stockpile_idx.get(sid,0):>6.3f} '
              f'{INCOME_LABELS[t]:>8}')

    # Pass 2: 列を追加
    print('\n[2/3] 書き出し中...')
    tmp_path = OUT_PATH.with_suffix('.tmp.parquet')
    writer = None
    for i in range(n_rg):
        rg = pf.read_row_group(i).to_pandas()
        keys = list(zip(
            rg['store_id'].astype(int).values,
            rg['dept_id'].astype(int).values,
            (rg['d_num'].astype(int) - 28).values,
        ))
        rg['te_store_dept_lag28'] = pd.Series(keys).map(te_lookup).astype('float32').values
        del keys
        sid_int = rg['store_id'].astype(int)
        rg['snap_dependency_score'] = sid_int.map(snap_dep).astype('float32').values
        rg['payroll_dependency_score'] = sid_int.map(payroll_dep).astype('float32').values
        rg['weekend_intensity'] = sid_int.map(we_intensity).astype('float32').values
        rg['luxury_affinity_score'] = sid_int.map(lux_affinity).astype('float32').values
        rg['price_sensitivity_index'] = sid_int.map(price_sens).astype('float32').values
        rg['pb_ratio'] = sid_int.map(pb_ratio).astype('float32').values
        rg['store_income_type'] = sid_int.map(store_income).astype('int8').values
        if 'snap_active' not in rg.columns:
            rg['snap_active'] = np.int8(0)
            rg.loc[rg['state_id'] == 0, 'snap_active'] = rg.loc[rg['state_id'] == 0, 'snap_CA'].values.astype('int8')
            rg.loc[rg['state_id'] == 1, 'snap_active'] = rg.loc[rg['state_id'] == 1, 'snap_TX'].values.astype('int8')
            rg.loc[rg['state_id'] == 2, 'snap_active'] = rg.loc[rg['state_id'] == 2, 'snap_WI'].values.astype('int8')
        if 'is_weekend' not in rg.columns:
            rg['is_weekend'] = (rg['wday'] >= 6).astype('int8')
        rg['snap_dep_interaction'] = (
            rg['snap_active'].values.astype('float32') * rg['snap_dependency_score'].values
        ).astype('float32')
        rg['weekend_interaction'] = (
            rg['is_weekend'].values.astype('float32') * rg['weekend_intensity'].values
        ).astype('float32')
        rg['snap_x_income'] = (
            rg['snap_active'].values.astype('int8') * 3
            + rg['store_income_type'].values.astype('int8')
        ).astype('int8')
        rg['price_x_psi'] = (
            rg['sell_price'].values.astype('float32') * rg['price_sensitivity_index'].values
        ).astype('float32')
        rg['snap_x_pb'] = (
            rg['snap_active'].values.astype('float32') * rg['pb_ratio'].values
        ).astype('float32')
        cie_keys = list(zip(
            rg['store_id'].astype(int).values,
            rg['cat_id'].astype(int).values,
        ))
        rg['cat_income_elasticity'] = (
            pd.Series(cie_keys).map(cie_lookup).fillna(1.0).astype('float32').values
        )
        del cie_keys
        rg['stockpiling_index'] = sid_int.map(stockpile_idx).fillna(0.0).astype('float32').values
        if 'roll_mean_56' in rg.columns:
            cie_vals = rg['cat_income_elasticity'].values
            rg['roll_mean_56_weighted'] = (
                rg['roll_mean_56'].values / np.where(cie_vals == 0, 1.0, cie_vals)
            ).astype('float32')
        else:
            rg['roll_mean_56_weighted'] = np.float32(0)
        # income_event_sensitivity
        rg['income_event_sensitivity'] = sid_int.map(income_event_sens).fillna(0).astype('int8').values
        # spike_hint
        snap_vals_j = rg['snap_active'].values.astype('int8')
        is_we_vals_j = rg['is_weekend'].values.astype('int8')
        day_vals_j = rg['day'].values if 'day' in rg.columns else np.int8(0)
        store_vals_j = rg['store_id'].astype(int).values
        cat_vals_j = rg['cat_id'].astype(int).values
        spike = np.zeros(len(rg), dtype='float32')
        for j in range(len(rg)):
            sid_j, cid_j = int(store_vals_j[j]), int(cat_vals_j[j])
            if int(snap_vals_j[j]) == 1:
                spike[j] = spike_expected.get((sid_j, cid_j, 0), 0.0)
            elif int(day_vals_j[j]) in PAYDAY_DAYS:
                spike[j] = spike_expected.get((sid_j, cid_j, 1), 0.0)
            elif int(is_we_vals_j[j]) == 1:
                spike[j] = spike_expected.get((sid_j, cid_j, 2), 0.0)
        rg['spike_hint'] = spike.astype('float32')
        # 旧列削除
        for old_col in ['snap_uplift_store']:
            if old_col in rg.columns:
                rg.drop(columns=[old_col], inplace=True)

        table = pa.Table.from_pandas(rg, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(tmp_path), table.schema)
        writer.write_table(table)
        del rg, table
        print(f'  rg {i+1}/{n_rg}')
    del pf
    if writer:
        writer.close()

    tmp_path.replace(OUT_PATH)
    print(f'Phase 1.5 完了 ({time.time()-t0:.0f}s, {OUT_PATH.stat().st_size/1e6:.0f} MB)')

## Step 2 (Phase 2): parquet → train/val/eval 分割

In [ ]:
# ============================================================
# Phase 2: parquet → train_X.dat, train_y.dat, train_cat.dat, val.parquet, eval.parquet
# ============================================================
VAL_START_DAY  = 1886
EVAL_START_DAY = 1914

TRAIN_X_PATH   = OUTPUT_DIR / 'train_X.dat'
TRAIN_Y_PATH   = OUTPUT_DIR / 'train_y.dat'
TRAIN_CAT_PATH = OUTPUT_DIR / 'train_cat.dat'
VAL_PATH       = OUTPUT_DIR / 'val.parquet'
EVAL_PATH      = OUTPUT_DIR / 'eval.parquet'

# FEATURES list from schema
pf = pq.ParquetFile(OUT_PATH)
all_cols = [f.name for f in pf.schema_arrow]
DROP_COLS = set(META_COLS + ['d', 'date', 'sales', 'wm_yr_wk', 'd_num',
                             'snap_CA', 'snap_TX', 'snap_WI'])
FEATURES = [c for c in all_cols if c not in DROP_COLS]
TARGET = 'sales'
n_rg = pf.metadata.num_row_groups

# --- 古い split ファイルとの整合性チェック ---
# val.parquet が存在する場合、特徴量列が parquet スキーマと一致するか確認
# 不一致なら古い split を削除して再生成
split_files = [TRAIN_X_PATH, TRAIN_Y_PATH, TRAIN_CAT_PATH, VAL_PATH, EVAL_PATH]
if VAL_PATH.exists():
    _val_cols = set(f.name for f in pq.ParquetFile(VAL_PATH).schema_arrow)
    _expected = set(FEATURES)
    if not _expected.issubset(_val_cols):
        _missing_feats = _expected - _val_cols
        print(f'[STALE] split ファイルに不足列: {_missing_feats}')
        print('  → 古い split ファイルを削除して再生成します')
        for p in split_files:
            if p.exists():
                p.unlink()
                print(f'    deleted: {p.name}')

if all(p.exists() for p in split_files):
    n_train = os.path.getsize(TRAIN_X_PATH) // (len(FEATURES) * 4)
    print(f'[SKIP] 分割済み (train: {n_train:,}, features: {len(FEATURES)})')
else:
    t0 = time.time()

    # Pass 1: count
    print('[1/3] train 行数カウント...')
    n_train = 0
    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=['d_num', 'lag_56']).to_pandas()
        n_train += int(((rg['d_num'] < VAL_START_DAY) & rg['lag_56'].notna()).sum())
        del rg
    print(f'  train: {n_train:,} rows, features: {len(FEATURES)}')

    X_mm = np.memmap(str(TRAIN_X_PATH), dtype='float32', mode='w+',
                     shape=(n_train, len(FEATURES)))
    y_mm = np.memmap(str(TRAIN_Y_PATH), dtype='float32', mode='w+',
                     shape=(n_train,))
    cat_mm = np.memmap(str(TRAIN_CAT_PATH), dtype='int8', mode='w+',
                       shape=(n_train,))

    # Pass 2: split
    print('[2/3] 分割中...')
    offset = 0
    val_writer = None
    eval_writer = None
    keep_cols_ve = ['id', 'cat_id', 'd_num', TARGET] + FEATURES

    for i in range(n_rg):
        rg = pf.read_row_group(i).to_pandas()

        # fillna 前にマスク計算 (lag_56 NaN 行を除外)
        mask_t = (rg['d_num'] < VAL_START_DAY) & rg['lag_56'].notna()
        rg[FEATURES] = rg[FEATURES].fillna(0)
        tr = rg[mask_t]
        if len(tr) > 0:
            n = len(tr)
            X_mm[offset:offset+n] = tr[FEATURES].values.astype('float32')
            y_mm[offset:offset+n] = tr[TARGET].values.astype('float32')
            cat_mm[offset:offset+n] = tr['cat_id'].values.astype('int8')
            offset += n

        mask_v = (rg['d_num'] >= VAL_START_DAY) & (rg['d_num'] < EVAL_START_DAY)
        v = rg[mask_v]
        if len(v) > 0:
            tbl = pa.Table.from_pandas(v[keep_cols_ve], preserve_index=False)
            if val_writer is None:
                val_writer = pq.ParquetWriter(str(VAL_PATH), tbl.schema)
            val_writer.write_table(tbl)

        mask_e = rg['d_num'] >= EVAL_START_DAY
        e = rg[mask_e]
        if len(e) > 0:
            tbl = pa.Table.from_pandas(e[keep_cols_ve], preserve_index=False)
            if eval_writer is None:
                eval_writer = pq.ParquetWriter(str(EVAL_PATH), tbl.schema)
            eval_writer.write_table(tbl)

        del rg, tr, v, e
        print(f'  rg {i+1}/{n_rg}  train: {offset:,}')

    X_mm.flush(); y_mm.flush(); cat_mm.flush()
    if val_writer: val_writer.close()
    if eval_writer: eval_writer.close()
    del X_mm, y_mm, cat_mm

    print(f'[3/3] 完了 ({time.time()-t0:.0f}s)')
    print(f'  train_X   : {os.path.getsize(TRAIN_X_PATH)/1e6:.0f} MB')
    print(f'  train_y   : {os.path.getsize(TRAIN_Y_PATH)/1e6:.0f} MB')
    print(f'  train_cat : {os.path.getsize(TRAIN_CAT_PATH)/1e6:.0f} MB')
    print(f'  val       : {os.path.getsize(VAL_PATH)/1e6:.0f} MB')
    print(f'  eval      : {os.path.getsize(EVAL_PATH)/1e6:.0f} MB')

del pf

# ============================================================
# 出力ファイル確認
# ============================================================
print(f'\n=== 出力ファイル ({OUTPUT_DIR}) ===')
for p in [OUT_PATH, TRAIN_X_PATH, TRAIN_Y_PATH, TRAIN_CAT_PATH, VAL_PATH, EVAL_PATH]:
    if p.exists():
        print(f'  {p.name:<25} {os.path.getsize(p)/1e6:>8.0f} MB')
    else:
        print(f'  {p.name:<25} MISSING')
print()
print('pipeline.ipynb で学習・評価・提出を実行してください')